In [ ]:
JOBS_GPU0 = ['icarl_full__nobn__superclass__s0']
JOBS_GPU1 = ['icarl_full__nobn__superclass__s1']
JOBS_CPU = []
RUN_TAG = 'vna4_m1_gir_live_task1_s01'
PRESET = 'M1_GIR_LIVE_TASK1_S01'
REPO_COMMIT = 'c6a81f3eed937a1f88a1b391d940ae58bb3096c0'
SOURCE_DIGEST = None
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000

GRAD_CLIP = None
K0 = 512
PLANNED_SEEDS = list(range(8))


# M1 task-1 live GIR: seeds 0 and 1
Fork of vna4_M1 on base 6be6543. T4×2: GPU0=s0, GPU1=s1. Same CIFAR-100 input and github_token Secret; Internet on. No source ZIP. K0 is the preregistered constant 512; no Gram authority or preflight is needed. Attach this pilot Output only when resuming it. Baselines and competitors are reused externally; this notebook trains only icarl_full with --gir-live-task1. Full 20 tasks, no clipping. GPU-hours and task-1 live/dead swap counts are printed at the end. Task-1 candidate Gram solves may be expensive; T4 runtime has not been measured. Deadline pauses preserve the complete task chain for a later session.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_m1_gir_live_task1_s01")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))

os.environ.pop('VNA_SOURCE_DIGEST', None)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from scripts.collect_parallel import validate_result
ROOT = STAGE_B / REPO_COMMIT[:12] / RUN_TAG
ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')

def terminal(job):
    folder = ROOT/job
    if not (folder/'summary.json').exists(): return False
    summary = json.loads((folder/'summary.json').read_text())
    if summary.get('status') not in ('COMPLETED','FAILED_DIVERGED'): return False
    protocol = json.loads((folder/'protocol.json').read_text())
    identity = summary['identity']
    seed = int(job.rsplit('__s',1)[1])
    expected = dict(code_commit=REPO_COMMIT, source_digest=None, arm='icarl_full',
        seed=seed, gir_live_task1=True, backbone='resnet32_nobn',
        class_order_kind='superclass', K0=512, solver='gram', curvature='power10',
        intervention_start_task=1, nobn_lr_scale=.5, expected_tasks=20)
    if (any(identity.get(k)!=v for k,v in expected.items())
            or any(protocol.get(k)!=v for k,v in identity.items())):
        raise ValueError('Pilot result identity mismatch: '+job)
    if summary['status']=='COMPLETED':
        validate_result(job, summary, protocol, summary['tasks'], REPO_COMMIT)
    elif not summary.get('diverged'):
        raise ValueError('Divergence coordinates missing: '+job)
    print('TERMINAL',job,summary['status'],summary.get('diverged'),flush=True)
    return True

def lane(gpu,jobs):
    for job in jobs:
        if pause_due(): return
        if terminal(job): continue
        seed = job.rsplit('__s',1)[1]
        cmd = [sys.executable,'-u','-m','experiments.main_cil','--arm','icarl_full',
            '--backbone','resnet32_nobn','--class-order','superclass','--seed',seed,
            '--nobn-lr-scale','0.5','--solver','gram','--curvature','power10',
            '--gir-live-task1','--max-tasks','20','--workers','2',
            '--device','cuda:0','--data-dir',str(DATA_ROOT),'--out',str(ROOT/job)]
        print('K0=512 (fixed); GPU',gpu,'seed',seed,'task1 GIR=d88abc4 live/dead; task2+=6be6543',flush=True)
        if not launch(cmd,gpu,job,RUN_ROOT/f'{RUN_TAG}_{job}.log'): return
        terminal(job)
        archive_outputs()

try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        jobs=[pool.submit(lane,0,JOBS_GPU0),pool.submit(lane,1,JOBS_GPU1)]
        for future in jobs: future.result()
finally:
    archive_stop.set();archiver.join(timeout=10);archive_outputs()


In [ ]:
rows=[]
for job in JOBS_GPU0+JOBS_GPU1:
    folder=ROOT/job
    summary=json.loads((folder/'summary.json').read_text()) if (folder/'summary.json').exists() else {}
    events=json.loads((folder/'interventions.json').read_text()) if (folder/'interventions.json').exists() else []
    swaps=[s for e in events if e.get('task')==1 for s in e.get('swap_records',[])]
    rows.append(dict(job=job,status=summary.get('status','PAUSED_OR_PENDING'),
        gpu_hours=summary.get('gpu_hours'),diverged=summary.get('diverged'),
        task1_live=sum(s['slot_kind']=='live' for s in swaps),
        task1_dead=sum(s['slot_kind']=='dead' for s in swaps),
        task1_interventions=sum(e.get('task')==1 for e in events)))
print(json.dumps(rows,indent=2))
(ROOT/'pilot_report.json').write_text(json.dumps(rows,indent=2))
archive_outputs()
if session_paused.is_set(): print('PAUSED: attach this notebook Output to resume exactly; keep the same pin and tag.')
if archive_errors: raise RuntimeError(archive_errors)
